In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from urllib.parse import unquote
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load CSV
file_path = '/content/drive/My Drive/all_test_public.csv'
df = pd.read_csv(file_path, header=None, low_memory=False)
sample_df = df.head(100).copy()

image_url_col = 10  # 11th column (0-based)

# Load BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Function to check if the image exists
def image_exists(url):
    try:
        if pd.isna(url) or not isinstance(url, str) or url.strip() == "":
            return False
        decoded_url = unquote(url)
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.head(decoded_url, timeout=5, headers=headers)
        return response.status_code == 200 and 'image' in response.headers.get('Content-Type', '')
    except:
        return False

# Function to generate a caption
def generate_caption(url):
    try:
        decoded_url = unquote(url)
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(decoded_url, timeout=10, headers=headers)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')
        inputs = processor(img, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        print(f"Error with URL {url}: {e}")
        return "Error generating caption"

# Add columns for image existence and captions
sample_df['image_exists'] = sample_df[image_url_col].apply(image_exists)

captions = []
for idx, row in sample_df.iterrows():
    url = row[image_url_col]
    if row['image_exists']:
        captions.append(generate_caption(url))
    else:
        captions.append("Image does not exist")

sample_df['caption'] = captions

# Save output
output_path = '/content/drive/My Drive/all_test_public_first_100_with_captions.csv'
sample_df.to_csv(output_path, index=False)

# Preview
print(sample_df[[image_url_col, 'image_exists', 'caption']].head())


                               10  image_exists               caption
0                       image_url         False  Image does not exist
1                             NaN         False  Image does not exist
2                             NaN         False  Image does not exist
3  http://i.imgur.com/cSIuEVF.jpg         False  Image does not exist
4                             NaN         False  Image does not exist


# Full image captioning

In [ ]:
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from urllib.parse import unquote
from transformers import BlipProcessor, BlipForConditionalGeneration

# Load CSV
file_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/polarity_results.csv'
df = pd.read_csv(file_path, header=None, low_memory=False)


image_url_col = 10  # 11th column (0-based)

# Load BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Function to check if the image exists
def image_exists(url):
    try:
        if pd.isna(url) or not isinstance(url, str) or url.strip() == "":
            return False
        decoded_url = unquote(url)
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.head(decoded_url, timeout=5, headers=headers)
        return response.status_code == 200 and 'image' in response.headers.get('Content-Type', '')
    except:
        return False

# Function to generate a caption
def generate_caption(url):
    try:
        decoded_url = unquote(url)
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(decoded_url, timeout=10, headers=headers)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content)).convert('RGB')
        inputs = processor(img, return_tensors="pt")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        return caption
    except Exception as e:
        print(f"Error with URL {url}: {e}")
        return "Error generating caption"

# Add columns for image existence and captions
df['image_exists'] = df[image_url_col].apply(image_exists)

captions = []
for idx, row in df.iterrows():
    url = row[image_url_col]
    if row['image_exists']:
        captions.append(generate_caption(url))
    else:
        captions.append("Image does not exist")

df['caption'] = captions

# Save output
output_path = '/content/drive/MyDrive/Research/Dataset/Fakeddit/all_samples (also includes non multimodal)/Original dataset/image-caption-added.csv'
df.to_csv(output_path, index=False)

# Preview
print(df[[image_url_col, 'image_exists', 'caption']].head())
